In [1]:
# Install required libraries (run once per environment)
#!pip install dataretrieval xarray rasterio regionmask geopandas shapely netCDF4 scikit-learn xgboost tensorflow-macos tensorflow-metal matplotlib seaborn

# --- Imports ---
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

from dataretrieval import nwis
from shapely.geometry import Point
import geopandas as gpd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import r2_score, mean_squared_error

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
import warnings
warnings.filterwarnings("ignore")


In [2]:
def get_usgs_streamflow(gage, start='1950-01-01', end='2020-12-31'):
    """Fetch daily streamflow (cfs) from USGS NWIS and compute March–October mean for each year."""
    df = nwis.get_record(sites=gage, service='dv', parameterCd='00060', start=start, end=end)
    df = df.reset_index()
    df['year'] = df['datetime'].dt.year
    df['month'] = df['datetime'].dt.month
    flow = (df[df['month'].between(3, 10)]
            .groupby('year')['00060_Mean']
            .mean()
            .reset_index(name='flow_cfs'))
    return flow

def get_gage_coordinates(gage):
    """Return (lat, lon) for a USGS gage ID using the current dataretrieval API."""
    info, metadata = nwis.get_info(sites=gage)  # unpack tuple
    # The first element is a DataFrame containing the site info.
    lat = float(info.loc[0, 'dec_lat_va'])
    lon = float(info.loc[0, 'dec_long_va'])
    return lat, lon


In [3]:
def get_scpdsi_grid(lat, lon, radius_deg=1.5, start_year=1950, end_year=2023):
    """
    Access scPDSI from TerraClimate hosted on AWS (Zarr format via pangeo-datastore).
    Extract cells within a radius around a coordinate and return March–October means.
    """
    # Load directly from cloud-optimized zarr store
    url = "https://data.chc.ucsb.edu/products/TerraClimate_zarr"
    ds = xr.open_zarr(f"{url}/scpdsi", consolidated=True)

    # Subset region & time range
    ds = ds.sel(lat=slice(lat + radius_deg, lat - radius_deg),
                lon=slice(lon - radius_deg, lon + radius_deg),
                time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))

    # Convert to DataFrame
    df = ds.to_dataframe().reset_index()

    # March–October mean
    df['year'] = pd.to_datetime(df['time']).dt.year
    df['month'] = pd.to_datetime(df['time']).dt.month
    df = df.query("month >= 3 and month <= 10")
    annual = df.groupby(['lat', 'lon', 'year'])['scpdsi'].mean().reset_index()
    return annual


In [4]:
def aggregate_scpdsi(df, method="mean"):
    """Aggregate multiple scPDSI grid cells into one representative time series."""
    if method == "mean":
        annual = df.groupby('year')['scpdsi'].mean().reset_index(name='mean_scpdsi')
    elif method == "maxcorr":
        raise NotImplementedError("maxcorr method would select cell most correlated with flow.")
    return annual


In [5]:
gage_id = "02361000"  # Choctawhatchee River near Newton, AL
flow = get_usgs_streamflow(gage_id)
lat, lon = get_gage_coordinates(gage_id)
pdsi_grid = get_scpdsi_grid(lat, lon, radius_deg=2.0, start_year=1921, end_year=2025)
pdsi_agg = aggregate_scpdsi(pdsi_grid)

# Merge datasets
df = pd.merge(flow, pdsi_agg, on="year").dropna()
print(df.head(), "\nYears of overlap:", len(df))


ValueError: unrecognized engine 'zarr' must be one of your download engines: ['netcdf4', 'scipy', 'store']. To install additional dependencies, see:
https://docs.xarray.dev/en/stable/user-guide/io.html 
https://docs.xarray.dev/en/stable/getting-started-guide/installing.html

In [ ]:
X = sm.add_constant(df['mean_scpdsi'])
y = df['flow_cfs']

model = sm.OLS(y, X).fit()
print(model.summary())

dw = durbin_watson(model.resid)
print(f"Durbin–Watson statistic: {dw:.2f}")

plt.figure(figsize=(6,5))
sns.regplot(x=model.fittedvalues, y=y, line_kws={'color':'red'})
plt.xlabel("Predicted Flow"); plt.ylabel("Observed Flow")
plt.title(f"SLR (No Lags): R²={model.rsquared:.2f}")
plt.show()


In [ ]:
X = sm.add_constant(df['mean_scpdsi'])
y = df['flow_cfs']

model = sm.OLS(y, X).fit()
print(model.summary())

dw = durbin_watson(model.resid)
print(f"Durbin–Watson statistic: {dw:.2f}")

plt.figure(figsize=(6,5))
sns.regplot(x=model.fittedvalues, y=y, line_kws={'color':'red'})
plt.xlabel("Predicted Flow"); plt.ylabel("Observed Flow")
plt.title(f"SLR (No Lags): R²={model.rsquared:.2f}")
plt.show()


In [ ]:
# Create lagged predictors
df['pdsi_lag1'] = df['mean_scpdsi'].shift(1)
df['pdsi_lag2'] = df['mean_scpdsi'].shift(2)
df = df.dropna(subset=['pdsi_lag1', 'pdsi_lag2'])

df.head()


In [ ]:
X = sm.add_constant(df[['mean_scpdsi', 'pdsi_lag1', 'pdsi_lag2']])
y = df['flow_cfs']

model_lag = sm.OLS(y, X).fit()
print(model_lag.summary())

dw = durbin_watson(model_lag.resid)
print(f"Durbin–Watson statistic: {dw:.2f}")

plt.figure(figsize=(6,5))
sns.regplot(x=model_lag.fittedvalues, y=y, line_kws={'color':'red'})
plt.xlabel("Predicted Flow"); plt.ylabel("Observed Flow")
plt.title(f"SLR with Lags: R²={model_lag.rsquared:.2f}")
plt.show()


In [ ]:
X = df[['mean_scpdsi', 'pdsi_lag1', 'pdsi_lag2']]
y = df['flow_cfs']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

results_lagged = {}

def eval_model(name, model):
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    r2 = r2_score(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results_lagged[name] = {'R2': r2, 'RMSE': rmse, 'preds': preds}
    print(f"{name}: R²={r2:.3f}, RMSE={rmse:.3f}")

# --- Run Models ---
eval_model("Random Forest (lags)", RandomForestRegressor(n_estimators=500, max_depth=6, random_state=42))
eval_model("XGBoost (lags)", XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=5, random_state=42))
eval_model("SVR (lags)", SVR(kernel='rbf', C=100, epsilon=0.1))
kernel = C(1.0) * RBF(length_scale=1.0)
eval_model("Gaussian Process (lags)", GaussianProcessRegressor(kernel=kernel, alpha=0.1))


In [ ]:
results_df = pd.DataFrame(results_lagged).T.sort_values("R2", ascending=False)
display(results_df)

best_model_name = results_df.index[0]
best_preds = results_lagged[best_model_name]['preds']

plt.figure(figsize=(10,5))
plt.plot(df['year'][-len(y_test):], y_test, label="Observed", color="black")
plt.plot(df['year'][-len(y_test):], best_preds, label=f"{best_model_name}", color="blue")
plt.title(f"Best Lagged Model: {best_model_name} | R²={results_lagged[best_model_name]['R2']:.2f}")
plt.xlabel("Year"); plt.ylabel("Flow (cfs)")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# Re-train best model (e.g., Random Forest) using all observed years
best_model = RandomForestRegressor(n_estimators=500, max_depth=6, random_state=42)
train = df.copy()

best_model.fit(train[['mean_scpdsi', 'pdsi_lag1', 'pdsi_lag2']], train['flow_cfs'])

# Prepare full scPDSI record
full_pdsi = aggregate_scpdsi(pdsi_grid)
full_pdsi['pdsi_lag1'] = full_pdsi['mean_scpdsi'].shift(1)
full_pdsi['pdsi_lag2'] = full_pdsi['mean_scpdsi'].shift(2)
full_pdsi = full_pdsi.dropna(subset=['pdsi_lag1', 'pdsi_lag2'])

# Predict reconstructed flow
full_pdsi['reconstructed_flow'] = best_model.predict(full_pdsi[['mean_scpdsi', 'pdsi_lag1', 'pdsi_lag2']])

# Plot extended reconstruction
plt.figure(figsize=(12,5))
plt.plot(full_pdsi['year'], full_pdsi['reconstructed_flow'], label='Reconstructed (ML, lags)', color='blue')
plt.plot(df['year'], df['flow_cfs'], label='Observed', color='black')
plt.title("Extended Streamflow Reconstruction using Lagged scPDSI (Random Forest)")
plt.xlabel("Year"); plt.ylabel("Flow (cfs)")
plt.legend(); plt.tight_layout(); plt.show()
